## Patient Journey - Lab Test
#### Data source used
* PHCDW.PHCDW_SANDBOX.IQV_QUEST_LAB
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT 

#### Steps:
1. Query UACR/eGFR lab test claims data after 2017-01-01
2. Get date level lab test stage records (forward fill for between two tests) for each patient
3. Save table to databricks

#### Notes:
1. Time frame: SVC_DT >= '2017-01-01' 
2. GFR test:  upper(LCL_LAB_TEST_NM) like '%GFR%' or upper(LCL_LAB_TEST_NM) like '%GLOMERULAR%' then 'GFR' 
3. UACR test:  upper(LCL_LAB_TEST_NM) like '%ALB%' or upper(LCL_LAB_TEST_NM) like '%RATIO%' then 'UACR'
4. GFR stage: 
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT <15 then 5
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=15 AND MEDIAN_TEST_RESULT <30 then 4
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <60 then 3
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=60 AND MEDIAN_TEST_RESULT <90 then 2
    * when test_type = 'GFR'  and MEDIAN_TEST_RESULT >=90 AND  MEDIAN_TEST_RESULT <=150 then 1
5. UACR stage:
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT <30 then 1
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT >=30 and MEDIAN_TEST_RESULT <300 then 2
    * when test_type = 'UACR' and MEDIAN_TEST_RESULT >=300 then 3
6. Check the the end_date of the data, now we set end_date_limit = Max(SVC_DT)

## NBA 3.1 Lab Query Optimization

**Goal:** make the 3.0 lab query dynamic so that adding a test, changing a rule or adjusting a setting is a file edit, not a SQL edit. The 3.0 query (cell "Load lab data") is kept untouched as a baseline. Four steps were added, each with its own query cell and a comparison cell that joins the new query with the previous one row by row (same patient, same day, same doctor, same test) so every change is verified.

### What was done

| Step | What moved out of the SQL | Where it lives now | Key result |
| --- | --- | --- | --- |
| 1 | Nothing. An inventory of the 15 hardcoded parts of the 3.0 query, grouped by where each fix would go (basket, config, code) | Markdown (cell 8) | Review only |
| 2 (items 1-5, 14) | Which LOINC codes count, test labels, units, valid range, stage cut-offs, and which result wins on the same day | Our copy of `Kerendia_Lab_Final.csv` in the Final Product Basket folder | UACR parts no longer staged; eGFR 150-200 = stage 1 instead of stage 0; same-day uses the preferred formula instead of the median of all variants |
| 3 (items 7-12) | Start and end dates, specialty filter, date formats, source table and column names | `LAB_CONFIG` block in cell 3a | Same output as Step 2, 19% less data pulled from Snowflake (dates applied inside the SQL instead of after) |
| 4 (items 6, 13, 15) | How result values are read (lab flags, above/below), patient ID rule, output columns | `VALUE_RULES` and `PATIENT_ID_RULE` in cell 4a | +35.7K eGFR test-days gained (2025+), nothing lost; a dropped-rows report on every run |

Each step keeps the previous step's changes. So Step 4 has everything: basket rules + config settings + code rules. The 3.0 query is never modified.

### What each step added

**Step 2 — Basket (cells 2a-2e)**
* Rewrote our copy of `Kerendia_Lab_Final.csv` to one row per code, with columns: `MATCH_TYPE` (LOINC or NAME), `MATCH_VALUE`, `TEST_TYPE`, `ACTIVE`, `ROLE` (RESULT or PART), `PRIORITY`, `UNITS` (pipe-separated), `VALID_MIN`, `VALID_MAX`, `STAGE_CUTOFFS` (e.g. `15:5|30:4|60:3|90:2|else:1`), `TEST_FAMILY`, `NOTE`. Sehar's original file is untouched.
* 21 LOINC codes (10 results, 11 parts) and 5 test-name backups, for tests GFR and UACR. Potassium and HbA1c are in the file with `ACTIVE = N`.
* Cell 2a reads the CSV, checks it (no duplicate codes, all rows of a test share the same rules, valid roles and match types), and builds `lab_test_rules`.
* Cell 2b is the query alone in its cell: `build_lab_query_basket(basket, rules, start_date)` fills the basket codes, name backups, unit lists, valid ranges and stage cut-offs into a SQL template. Start date and specialty list are still the 3.0 values (handled in Step 3).
* Cell 2c runs it with the same post-processing as the 3.0 cell (HCP universe join, validation filter).
* Cell 2d compares with the 3.0 query: Part A joins both queries in Snowflake (all US, 2025-01 to 2026-09); Part B does validation HCPs month by month with `to_patient_months` and `summary` helper functions defined here and reused in 3d and 4d.

**Step 3 — Config (cells 3a-3e)**
* Cell 3a defines `LAB_CONFIG`: `start_date` (from `DATA_START_DATE`), `end_date` (from `date_end`), `specialties` (the 3.0 list of 10 primary-care codes, or `None` for no filter), `date_formats` (`MM/DD/YYYY`, `DDMONYYYY`), and `source` / `provider` blocks with table names and column-name mappings.
* Checks that the source and provider tables and all mapped columns exist before building the query.
* Cell 3b is the query alone: `build_lab_query(basket, rules, cfg)` adds dates, specialty filter, date parsing (`TRY_TO_DATE` with fallback), and table/column names from `LAB_CONFIG` to the Step 2 template.
* Cell 3d compares with Step 2 (Part A identical, Part B identical, Part C rows pulled 312.9M vs 252.2M, Part D date format check: 38 empty dates, 0 unknown formats).

**Step 4 — Code (cells 4a-4e)**
* Cell 4a defines `VALUE_RULES` (remove `^` flags, keep `>` / `<` when the stage is certain), `PATIENT_ID_RULE` (same 3.0 rule: digits only), and `lab_rules_step4` (per-test rules read from the basket, with cut-offs sorted low to high). Includes `sql_value()` and `sql_patient_id()` helpers that generate the SQL for value parsing and patient ID validation.
* Cell 4b is the query alone: `build_lab_query_step4(basket, rules, cfg)` adds shared value and patient ID rules, a `QUAL` column (`=`, `>`, `<`), an `ABOVE_BELOW_VALUE` output column, and a `lab_drop_report_step4` SQL that reuses the same CTEs to count every source row under the first rule it fails.
* Cell 4c runs it. Output columns follow the basket: one stage column per test with cut-offs (today `GFR_STAGE`, `UACR_STAGE`).
* Cell 4d compares with Step 3 (Part A all US, Part B validation HCPs, Part C dropped-rows report).

### Findings

**What "lost" and "gained" mean:** each comparison joins both queries on the same patient, day, doctor and test. "Lost" = the old query returned a row but the new one did not (a rule excluded it). "Gained" = the new query returned a row the old one dropped.

**Step 2 vs 3.0 (all US, 2025-01 to 2026-09)**

| | eGFR | UACR |
| --- | --- | --- |
| Same stage | 106.73M test-days | 19.81M test-days |
| Different stage (one step) | 55K (0.05%) | 27K (0.14%) |
| Lost | 260 (212 above 200, 48 = 0) | 68.4K (parts) + 2.5K (above 10000) |
| Gained | 0 | 0 |

No gains because the test-name backup keeps the 3.0 rule for rows without a LOINC code, and other filters are still the 3.0 ones at this step.

**Step 3 vs Step 2 (all US, 2025-01 to 2026-09)**

| | eGFR | UACR |
| --- | --- | --- |
| Identical | 107.17M | 19.86M |
| Different / lost / gained | 0 | 0 |
| Rows pulled from Snowflake | 312.9M | -- |
| Step 3 rows pulled | 252.2M (-19%) | -- |

**Step 4 vs Step 3 (all US, 2025-01 to 2026-09)**

| | eGFR | UACR |
| --- | --- | --- |
| Identical | 107.17M | 19.86M |
| Gained: above/below with a clear stage (`>90`, `<4`) | +35.7K (26.8K patients) | +535 |
| Gained: lab flag or leading dot (`75^`, `.5`) | 0 | +747 |
| Lost | 0 | 0 |
| Different stage | 1 | 0 |

**One-off check: 2023-2024** (same comparison run from chat, not in 4d)

| | eGFR | UACR |
| --- | --- | --- |
| Gained: lab-flag values (`75^`, only in 2023 data) | +75.7K (71K patients) | +10.1K |
| Gained: above/below with a clear stage | +568 | +568 |
| Different stage | 3 | 0 |

**Validation HCPs (10 doctors, full date range 2023-01 to 2026-09)**

No change at any step. Same 225 patients, 878 eGFR test-days, 289 UACR test-days, 6,139 patient-months, 2,347 months with eGFR stage 3-5.

**Dropped-rows report** (all US, 2023-01 to 2026-09, first reason that fails). The 3.0 query also drops all these rows silently; Step 4 makes it visible.

| Reason | eGFR rows | UACR rows | Recoverable? |
| --- | --- | --- | --- |
| Patient ID empty or -1 | 65.6M | 11.8M | No |
| Service date empty or unreadable | 27 | 11 | No |
| NPI missing | 24.3M | 3.8M | No |
| Part of a test, not a result | 0 | 16.0M | No (deliberate) |
| Value empty | 1.6M | 0.8M | No |
| Value is text (DNR, NOTE, See Comment) | 10.8M | 6.0M | No |
| Unit not allowed | 0 | 32.8K | Possibly (add to basket) |
| Outside valid range | 4.6M | 26.2K | No |
| Above/below, stage unclear (`>60`) | 827K | 314 | Maybe (decision) |
| Specialty not in the 10-code list | 211.4M | 21.2M | Yes (decision) |
| Kept | 631.9M | 106.9M | -- |

### Decisions already made

| Decision | Choice | Rationale |
| --- | --- | --- |
| eGFR valid max | 200 (was 150) | Values 150-200 are stage 1, not stage 0; keeps the one validation patient with only an eGFR of 153 |
| Basket file structure | One row per code, all rules as columns | Adding a code or changing a rule is a CSV edit; Sehar's original untouched |
| Test-name backup | Kept the 3.0 patterns | Rows with no LOINC code are still found by name, so nothing is lost |
| Same-day priority | Lowest PRIORITY number wins; exact value beats above/below | Stops mixing Black and non-Black eGFR on the same day |
| Lab flags (^) | Removed before reading the number | `75^` reads as 75, not a text value to drop |
| Above/below staging | Kept only when the whole range has one stage | `>90` = stage 1, `<15` = stage 5, `<30` = A1; `>60` is not staged (stage 1 or 2 unclear) |

### Decisions for the final review

| Decision | Options | Data behind it |
| --- | --- | --- |
| Specialty filter | (a) keep the 10 primary-care codes, (b) remove the filter, (c) add cardiology, endocrinology, nephrology | Removing it adds ~16% eGFR patients and ~13% UACR patients. The report shows it cuts 211M eGFR rows, the largest single cut, and this is real patient data |
| Where LAB_CONFIG lives | (a) stay in cell 3a of this notebook, (b) move to `00_cohort_config` | Moving it shares the settings with other patient-journey notebooks |
| `>60` eGFR (827K rows) | (a) keep it out (as today), (b) count it as "not CKD stage 3+" | It means eGFR is 60 or more, so stage 1 or 2. Keeping it out is conservative; using it as "not stage 3+" adds patients very unlikely to be CKD stage 3+ |

### What is NOT done yet

Section 2 ("Data cleanup and processing", cells 26-32) still uses `df_lab` from the 3.0 cell. Once the decisions above are made, switching to `df_lab_step4` is a one-variable change. The new features (potassium, HbA1c) are in the basket CSV with `ACTIVE = N`; activating them means changing to `Y`. The notebook and basket CSV are not in a Git folder yet.

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

In [0]:
# Source: the 3.0 "Pat Journey Lab" query (00_Write_Input_Files), run directly from Snowflake.
# The pre-built kerendia_nba_3_0_lab_query_output table was removed in the 2026-10-26 refresh,
# so this notebook now pulls it itself (same pattern as 03_patient_journey_rx / 04_t2d_flag).
lab_query = """
      SELECT
          PATIENT_ID,
          SVC_DT_STD AS DATE,
          NPI_ID AS NPI_NUMBER,
          TEST_TYPE,
          MEDIAN(LCL_LAB_TEST_VAL_TXT_FLOAT) as MEDIAN_TEST_RESULT,
          CASE WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT <15 THEN 5
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=15 AND MEDIAN_TEST_RESULT <30 THEN 4
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <60 THEN 3
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=60 AND MEDIAN_TEST_RESULT <90 THEN 2
          WHEN TEST_TYPE = 'GFR' AND MEDIAN_TEST_RESULT >=90 AND MEDIAN_TEST_RESULT <=150 THEN 1
          WHEN TEST_TYPE = 'UACR' THEN NULL
          ELSE 0
          END AS GFR_STAGE,
          CASE WHEN TEST_TYPE = 'UACR' AND MEDIAN_TEST_RESULT <30 THEN 1
          WHEN TEST_TYPE = 'UACR' AND MEDIAN_TEST_RESULT >=30 AND MEDIAN_TEST_RESULT <300 THEN 2
          WHEN TEST_TYPE = 'UACR'AND MEDIAN_TEST_RESULT >=300 THEN 3
          WHEN MEDIAN_TEST_RESULT IS NULL THEN NULL
          WHEN TEST_TYPE = 'GFR' THEN NULL
          ELSE 0
          END AS UACR_STAGE
          FROM (
              SELECT DISTINCT CAST(CAST(PATIENT_ID AS INT) AS VARCHAR(64)) AS PATIENT_ID, NPI_ID , SVC_DT_STD, TEST_TYPE,
              TRY_TO_DOUBLE(TRIM(LCL_LAB_TEST_VAL_TXT)) AS LCL_LAB_TEST_VAL_TXT_FLOAT  FROM
              (
                SELECT *,
                CASE
                  WHEN UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%GFR%' OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%GLOMERULAR%' THEN 'GFR'
                  WHEN UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%ALB%' OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE '%RATIO%' THEN 'UACR'
                  ELSE 'OTHER'
                END AS TEST_TYPE,
                CASE
                  WHEN SVC_DT LIKE '%/%/%' THEN TO_DATE(SVC_DT,'MM/DD/YYYY')
                  ELSE TO_DATE(LEFT(SVC_DT,9),'DDMONYYYY')
                END AS SVC_DT_STD
                FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST
                WHERE REGEXP_LIKE(PATIENT_ID, '^[0-9]+$')
                AND LCL_LAB_TEST_VAL_TXT IS NOT NULL
                AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%-%'
                AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%>%'
                AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%<%'
                AND REGEXP_LIKE(TRIM(LCL_LAB_TEST_VAL_TXT), '^[0-9]+(\\.[0-9]+)?$')
              ) lab1
              LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR dct
              ON lab1.NPI_ID = dct.NPI_NUMBER
              WHERE UPPER(REPLACE(LCL_LAB_TEST_RSLT_UOM_CD_CLEAN, ' ','')) IN
              ('ML/MIN/1.73M2', 'MCG/MGCREAT','MG/GCREAT','MG/GCR','MG/G{CREAT}')
              AND PRIMARY_SPECIALTY_CODE IN ('FM', 'FPG', 'GP', 'GPM', 'IFP', 'IM', 'IMG', 'IPM', 'NRP', 'PHA')
              AND TEST_TYPE <> 'OTHER'
              AND SVC_DT_STD >= '2021-11-30'
              AND NPI_ID IS NOT NULL
          ) LAB
      GROUP BY 1, 2, 3, 4
"""

df_lab = Get_Data_Snowflakes(lab_query)
df_lab = df_lab.filter(col("DATE") <= date_end)

# HCP universe (same as 3.0 00_Write_Input_Files)
_hcp_universe = (spark.read.format("delta").load(hcp_universe_path).select("PROVIDER_ID").distinct()
                 .join(spark.read.format("delta").load(provider_npi_bhid_mapping_path).select("PROVIDER_ID", "NPI_NUMBER"),
                       on="PROVIDER_ID", how="left"))
df_lab = df_lab.join(_hcp_universe, how="inner", on="NPI_NUMBER")

# --- original 3.1 logic from here ---
df_lab = df_lab.withColumnRenamed("DATE", "SVC_DT")

# Filter to data >= DATA_START_DATE (from config)
df_lab = df_lab.filter(F.col("SVC_DT") >= DATA_START_DATE)

# Validation filter: 10 test HCPs (from config)
df_lab, test_patients = apply_hcp_filter(df_lab)

df_lab = df_lab.select("PATIENT_ID", "SVC_DT", "GFR_STAGE", "UACR_STAGE")

df_lab = df_lab.withColumn("PATIENT_ID", col("PATIENT_ID").cast('string'))

df_lab.display()

### Step 1: manual (hardcoded) parts of the 3.0 lab query

Review of the query in the cell above ("Load lab data"). Nothing has been changed yet; each item is a candidate to make dynamic.

| # | Hardcoded today | Risk if the data changes | Dynamic option | Where it would live |
| --- | --- | --- | --- | --- |
| 1 | **Which tests count:** name patterns `%GFR%`, `%GLOMERULAR%` (eGFR) and `%ALB%`, `%RATIO%` (UACR) | Renamed tests are lost without notice; other tests with "ALB" or "RATIO" in the name get in | List the codes (LOINC) per test in the basket, with the names kept as a backup | Basket |
| 2 | **Test labels** `'GFR'` / `'UACR'` written into every CASE | Adding a test (e.g. potassium, HbA1c) means editing the SQL in several places | Read the test families from the basket and generate the CASE | Basket |
| 3 | **Allowed units:** one list for both tests (`ML/MIN/1.73M2`, `MG/GCREAT`, ...) | A new unit spelling is dropped without notice, and a unit is accepted for the wrong test | One unit list per test family | Basket |
| 4 | **Stage cut-offs:** 15/30/60/90 for eGFR, 30/300 for UACR | Changing a cut-off (e.g. splitting stage 3 into 3a/3b) means editing the SQL | A small cut-off table per test | Basket or config |
| 5 | **Valid range:** eGFR above 150 becomes stage 0, and anything else that fails the CASE also becomes 0 | Bad values get a stage (an eGFR of 0 becomes stage 5) | A min/max per test | Basket |
| 6 | **Value format:** plain numbers only; any `-`, `>`, `<` is dropped | A change in how results are reported (`92^`, `>60`) loses rows without notice | Parsing rules kept in one place | Code (one shared function) |
| 7 | **Specialty filter:** `FM, FPG, GP, GPM, IFP, IM, IMG, IPM, NRP, PHA` | Can't add cardiology, endocrinology or nephrology for HF / T1D without editing the SQL | A configurable list, or no filter | Config |
| 8 | **Start date** `'2021-11-30'` in the SQL, plus a second filter later on `DATA_START_DATE` | Two dates that can disagree | Use `DATA_START_DATE` only, inside the SQL | Config |
| 9 | **End date:** `date_end` is applied in Spark after pulling the full history | Pulls more data than needed (slow) | Pass `date_end` into the SQL | Config |
| 10 | **Date formats:** `MM/DD/YYYY` and `DDMONYYYY` written into a CASE | A new format gives an empty date and the row is dropped without notice | Safe parsing (`TRY_TO_DATE`) plus a format list; flag rows it can't parse | Config / code |
| 11 | **Source tables:** the Quest table and `DIM_PRVDR` names written into the SQL | A refresh or rename breaks the query; adding LabCorp means a second copy of the query | A source list (table, lab name) | Config |
| 12 | **Source column names** (`LCL_LAB_TEST_NM_CLEAN`, `LCL_LAB_TEST_VAL_TXT`, `SVC_DT`, `NPI_ID`, ...) | If the vendor renames a column, every reference must be edited | One column mapping per source | Config |
| 13 | **Patient ID rules:** numeric-only regex + cast to INT | A change in ID format drops rows without notice | One shared rule, plus a count of rows dropped | Code |
| 14 | **Same-day aggregation:** `MEDIAN` over all results per patient, day, NPI and test | Mixes different equations (Black and non-Black eGFR) | A priority column per code in the basket (which result to prefer) | Basket |
| 15 | **Output columns:** only the stages are kept | New features (value, test code) mean editing the select | The output column list follows the basket test families | Code |

**Grouped by where the change would live**
* **Basket (codes and test rules):** items 1-5 and 14. This is what makes "add a code once and it flows downstream" work.
* **Config (run settings):** items 7-12. Values changed between runs, with no code edits.
* **Code (written once, shared):** items 6, 13 and 15.

Nothing that is dropped without notice is reported today (items 3, 6, 10, 13). Each of those could print a count of the rows it drops, so data loss shows up on every run.

## Step 2: Basket changes (items 1-5 and 14)

The 3.0 lab query is rebuilt so that all test rules come from the lab basket file `Final Product Basket/Kerendia_Lab_Final.csv` (our copy): which codes count, test labels, units, valid range, stage cut-offs and which result wins on the same day. Adding or changing a code or a rule is now a CSV edit, with no code change.

Everything else is kept exactly as in 3.0 (dates, specialty filter, value format, patient ID rules), so the comparison shows only the effect of the basket.

| Cell | What it does |
| --- | --- |
| 2a | Reads the basket and checks it is valid |
| 2b | **The lab query** (built from the basket) |
| 2c | Runs the query with the same post-processing as the 3.0 cell |
| 2d | Compares the 3.0 query with the basket query (the evidence for 2e) |
| 2e | Findings: what changed compared with the 3.0 query |

In [0]:
# 2a. Read the lab basket and check it is valid before building the query.
#
# Basket file: one row per code (MATCH_TYPE = LOINC) or per test-name backup (MATCH_TYPE = NAME)
#   TEST_TYPE      GFR / UACR / ...           ACTIVE    Y = used in this journey
#   ROLE           RESULT = staged, PART = part of a test (creatinine, albumin...), never staged
#   PRIORITY       several results same patient/day: only the lowest number is used
#   UNITS          allowed units, separated by |          VALID_MIN / VALID_MAX   accepted values
#   STAGE_CUTOFFS  value:stage pairs, e.g. 15:5|30:4|60:3|90:2|else:1 (value below 15 -> stage 5 ...)
import pandas as pd

LAB_BASKET_PATH = ("/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sm/NBA 3.1 - Dev/03_Feature_Engineering/03_HCP_level_features/Final Product Basket/Kerendia_Lab_Final.csv")

lab_basket = pd.read_csv(LAB_BASKET_PATH, dtype=str).fillna("")
lab_basket = lab_basket[lab_basket["ACTIVE"] == "Y"]

# Checks: the cell stops with a clear message if the file is not valid
_test_cols = ["UNITS", "VALID_MIN", "VALID_MAX", "STAGE_CUTOFFS"]
_results = lab_basket[lab_basket["ROLE"] == "RESULT"]
_inconsistent = _results.groupby("TEST_TYPE")[_test_cols].nunique().gt(1).any(axis=1)
assert not _inconsistent.any(), f"Rows of the same test have different rules: {list(_inconsistent[_inconsistent].index)}"
assert set(lab_basket["ROLE"]) <= {"RESULT", "PART"}, "ROLE must be RESULT or PART"
assert set(lab_basket["MATCH_TYPE"]) <= {"LOINC", "NAME"}, "MATCH_TYPE must be LOINC or NAME"
assert not lab_basket[lab_basket["MATCH_TYPE"] == "LOINC"]["MATCH_VALUE"].duplicated().any(), "A LOINC code is listed twice"

# One set of rules per test, read from the basket
lab_test_rules = {}
for t, r in _results.groupby("TEST_TYPE").first().iterrows():
    pairs = [p.split(":") for p in r["STAGE_CUTOFFS"].split("|")]
    lab_test_rules[t] = {"units": r["UNITS"].split("|"), "valid_min": r["VALID_MIN"], "valid_max": r["VALID_MAX"],
                         "cuts": [(v, s) for v, s in pairs if v != "else"],
                         "top_stage": [s for v, s in pairs if v == "else"][0]}

_codes = lab_basket[lab_basket["MATCH_TYPE"] == "LOINC"]
_names = lab_basket[lab_basket["MATCH_TYPE"] == "NAME"]
print(f"Basket OK: {len(_codes)} codes ({(_codes.ROLE == 'RESULT').sum()} results, {(_codes.ROLE == 'PART').sum()} parts) "
      f"and {len(_names)} name backups, for tests {sorted(lab_test_rules)}")
display(pd.DataFrame(lab_test_rules).T.reset_index().rename(columns={"index": "TEST_TYPE"}).astype(str))

In [0]:
# 2b. The lab query, built from the basket (2a). Same structure and row filters as the 3.0 query;
# codes, test labels, units, valid range, stage cut-offs and same-day priority come from the basket file.
# Start date and specialty list are still as in 3.0 (config step later).
LAB_START_DATE = "2021-11-30"   # as in 3.0 (config step later)
LAB_SPECIALTIES = ["FM", "FPG", "GP", "GPM", "IFP", "IM", "IMG", "IPM", "NRP", "PHA"]   # as in 3.0 (config step later)

def _q(values):
    return ", ".join(f"'{v}'" for v in values)

def build_lab_query_basket(basket, rules, start_date):
    codes = basket[basket["MATCH_TYPE"] == "LOINC"]
    names = basket[basket["MATCH_TYPE"] == "NAME"]
    # name backups: if several patterns match, the most specific (longest) one wins
    names = names.assign(_len=names["MATCH_VALUE"].str.len()).sort_values("_len", ascending=False)
    code_values = ",\n        ".join(
        f"('{r.MATCH_VALUE}', '{r.TEST_TYPE}', '{r.ROLE}', {r.PRIORITY or 'NULL'})" for r in codes.itertuples())
    name_type = " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN '{r.TEST_TYPE}'" for r in names.itertuples())
    name_role = " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN '{r.ROLE}'" for r in names.itertuples())
    name_prio = " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN {r.PRIORITY or 'NULL'}" for r in names.itertuples())
    valid = "\n           OR ".join(
        f"(TEST_TYPE = '{t}' AND UOM IN ({_q(r['units'])}) AND TEST_NUM BETWEEN {r['valid_min']} AND {r['valid_max']})"
        for t, r in rules.items())
    stages = ",\n  ".join(
        f"CASE WHEN TEST_TYPE = '{t}' THEN CASE "
        + " ".join(f"WHEN MEDIAN_TEST_RESULT < {c} THEN {s}" for c, s in r["cuts"])
        + f" ELSE {r['top_stage']} END END AS {t}_STAGE"
        for t, r in rules.items())
    return f"""
WITH basket AS (
  SELECT * FROM VALUES
        {code_values}
  AS b(LOINC_CD, TEST_TYPE, ROLE, PRIORITY)
),
src AS (   -- same row filters as 3.0
  SELECT CAST(CAST(PATIENT_ID AS INT) AS VARCHAR(64)) AS PATIENT_ID,
         NPI_ID,
         CASE WHEN SVC_DT LIKE '%/%/%' THEN TO_DATE(SVC_DT, 'MM/DD/YYYY')
              ELSE TO_DATE(LEFT(SVC_DT, 9), 'DDMONYYYY') END AS SVC_DT_STD,
         LOINC_CD,
         UPPER(LCL_LAB_TEST_NM_CLEAN) AS TEST_NM,
         UPPER(REPLACE(LCL_LAB_TEST_RSLT_UOM_CD_CLEAN, ' ', '')) AS UOM,
         TRY_TO_DOUBLE(TRIM(LCL_LAB_TEST_VAL_TXT)) AS TEST_NUM
  FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST
  WHERE REGEXP_LIKE(PATIENT_ID, '^[0-9]+$')
    AND LCL_LAB_TEST_VAL_TXT IS NOT NULL
    AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%-%'
    AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%>%'
    AND TRIM(LCL_LAB_TEST_VAL_TXT) NOT ILIKE '%<%'
    AND REGEXP_LIKE(TRIM(LCL_LAB_TEST_VAL_TXT), '^[0-9]+(\\.[0-9]+)?$')
    AND (LOINC_CD IN (SELECT LOINC_CD FROM basket) OR UPPER(LCL_LAB_TEST_NM_CLEAN) LIKE ANY ({_q(names['MATCH_VALUE'])}))
),
typed AS (   -- basket first; test name only as backup for codes not in the basket
  SELECT s.PATIENT_ID, s.NPI_ID, s.SVC_DT_STD, s.LOINC_CD, s.UOM, s.TEST_NUM,
         COALESCE(b.TEST_TYPE, CASE {name_type} END) AS TEST_TYPE,
         COALESCE(b.ROLE, CASE {name_role} END) AS ROLE,
         IFF(b.LOINC_CD IS NULL, 'NAME', 'BASKET') AS MATCHED_BY,
         COALESCE(b.PRIORITY, CASE {name_prio} END) AS PRIORITY
  FROM src s LEFT JOIN basket b ON s.LOINC_CD = b.LOINC_CD
),
kept AS (    -- same specialty / date / NPI filters as 3.0; units and valid range per test from the basket
  SELECT t.*
  FROM typed t
  LEFT JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR dct ON t.NPI_ID = dct.NPI_NUMBER
  WHERE t.ROLE = 'RESULT'
    AND ({valid})
    AND dct.PRIMARY_SPECIALTY_CODE IN ({_q(LAB_SPECIALTIES)})
    AND t.SVC_DT_STD >= '{start_date}'
    AND t.NPI_ID IS NOT NULL
  QUALIFY PRIORITY = MIN(PRIORITY) OVER (PARTITION BY t.PATIENT_ID, t.NPI_ID, t.SVC_DT_STD, t.TEST_TYPE)
),
vals AS (    -- same DISTINCT as 3.0 before the median
  SELECT DISTINCT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE, TEST_NUM FROM kept
),
info AS (
  SELECT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE,
         LISTAGG(DISTINCT COALESCE(LOINC_CD, 'NO_LOINC'), ',') AS LOINC_CDS,
         MIN(MATCHED_BY) AS MATCHED_BY
  FROM kept GROUP BY 1, 2, 3, 4
),
agg AS (
  SELECT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE, MEDIAN(TEST_NUM) AS MEDIAN_TEST_RESULT
  FROM vals GROUP BY 1, 2, 3, 4
)
SELECT a.PATIENT_ID, a.SVC_DT_STD AS DATE, a.NPI_ID AS NPI_NUMBER, a.TEST_TYPE, a.MEDIAN_TEST_RESULT,
  {stages},
  i.LOINC_CDS, i.MATCHED_BY
FROM agg a JOIN info i USING (PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE)
"""

lab_query_basket = build_lab_query_basket(lab_basket, lab_test_rules, LAB_START_DATE)
print(lab_query_basket)


In [0]:
# 2c. Run the basket query (2b) with the same post-processing as the 3.0 cell "Load lab data"
df_lab_basket_all = Get_Data_Snowflakes(lab_query_basket)
df_lab_basket_all = (df_lab_basket_all
    .filter(col("DATE") <= date_end)
    .join(_hcp_universe, how="inner", on="NPI_NUMBER")
    .withColumnRenamed("DATE", "SVC_DT")
    .filter(F.col("SVC_DT") >= DATA_START_DATE)
    .withColumn("GFR_STAGE", col("GFR_STAGE").cast("int"))
    .withColumn("UACR_STAGE", col("UACR_STAGE").cast("int")))
df_lab_basket_all, test_patients_basket = apply_hcp_filter(df_lab_basket_all)

# Same 4 columns as df_lab, so section 2 can use it unchanged once approved
df_lab_basket = df_lab_basket_all.select("PATIENT_ID", "SVC_DT", "GFR_STAGE", "UACR_STAGE")

display(df_lab_basket_all.orderBy("PATIENT_ID", "SVC_DT"))

In [0]:
# 2d. Compare the 3.0 query (cell "Load lab data") with the basket query (2b / 2c).
#     Part A: all US patients, result by result (same patient, same day, same test)
#     Part B: the 10 validation HCPs, month by month (as section 2 uses it)
from pyspark.sql import Window

CMP_START = "2025-01-01"
_gfr, _uacr = lab_test_rules["GFR"], lab_test_rules["UACR"]

# ---------------------------------------------------------------------------
# Part A: all US patients. Both queries run side by side in Snowflake.
# ---------------------------------------------------------------------------
_compare_sql = f"""
WITH old AS (
  SELECT PATIENT_ID, DATE, TEST_TYPE,
         MAX(COALESCE(GFR_STAGE, UACR_STAGE)) AS STAGE_OLD, MAX(MEDIAN_TEST_RESULT) AS VALUE_OLD
  FROM ({lab_query}) WHERE DATE BETWEEN '{CMP_START}' AND '{date_end}' GROUP BY 1, 2, 3
),
new AS (
  SELECT PATIENT_ID, DATE, TEST_TYPE, MAX(COALESCE(GFR_STAGE, UACR_STAGE)) AS STAGE_NEW
  FROM ({lab_query_basket}) WHERE DATE BETWEEN '{CMP_START}' AND '{date_end}' GROUP BY 1, 2, 3
),
both AS (
  SELECT COALESCE(o.TEST_TYPE, n.TEST_TYPE) AS TEST_TYPE, COALESCE(o.PATIENT_ID, n.PATIENT_ID) AS PATIENT_ID,
         o.STAGE_OLD, n.STAGE_NEW, o.VALUE_OLD
  FROM old o FULL OUTER JOIN new n USING (PATIENT_ID, DATE, TEST_TYPE)
)
SELECT TEST_TYPE,
  CASE WHEN STAGE_OLD = STAGE_NEW THEN '1. same stage'
       WHEN STAGE_OLD IS NOT NULL AND STAGE_NEW IS NOT NULL THEN '2. different stage'
       WHEN STAGE_OLD IS NULL THEN '3. only basket query (gained)'
       ELSE '4. only 3.0 query (lost)' END AS RESULT,
  CASE WHEN RESULT LIKE '2.%' THEN 'stage ' || STAGE_OLD || ' -> ' || STAGE_NEW
       WHEN RESULT LIKE '4.%' AND TEST_TYPE = 'GFR' AND VALUE_OLD > {_gfr['valid_max']} THEN 'eGFR above {_gfr['valid_max']} (3.0 gave stage 0)'
       WHEN RESULT LIKE '4.%' AND TEST_TYPE = 'GFR' AND VALUE_OLD < {_gfr['valid_min']} THEN 'eGFR below {_gfr['valid_min']} (3.0 gave stage 5)'
       WHEN RESULT LIKE '4.%' AND TEST_TYPE = 'UACR' AND VALUE_OLD > {_uacr['valid_max']} THEN 'UACR above {_uacr['valid_max']}'
       WHEN RESULT LIKE '4.%' AND TEST_TYPE = 'UACR' THEN 'only a part of the test (e.g. urine creatinine), not a ratio'
       WHEN RESULT LIKE '4.%' THEN 'other'
  END AS DETAIL,
  COUNT(*) AS TEST_DAYS, COUNT(DISTINCT PATIENT_ID) AS PATIENTS
FROM both GROUP BY 1, 2, 3 ORDER BY 1, 2, 4 DESC
"""
print(f"A) All US patients, {CMP_START} to {date_end}: patient x day x test")
display(Get_Data_Snowflakes(_compare_sql).orderBy("TEST_TYPE", "RESULT", F.desc("TEST_DAYS")))

# ---------------------------------------------------------------------------
# Part B: validation HCPs, month by month (stage carried forward until the next test)
# ---------------------------------------------------------------------------
def to_patient_months(df):
    df = df.withColumn("MONTH", F.trunc("SVC_DT", "month"))
    per_month = df.groupBy("PATIENT_ID", "MONTH").agg(
        F.max_by("GFR_STAGE", F.when(F.col("GFR_STAGE").isNotNull(), F.col("SVC_DT"))).cast("int").alias("GFR"),
        F.max_by("UACR_STAGE", F.when(F.col("UACR_STAGE").isNotNull(), F.col("SVC_DT"))).cast("int").alias("UACR"))
    months = (per_month.groupBy("PATIENT_ID").agg(F.min("MONTH").alias("FIRST"))
              .select("PATIENT_ID", F.explode(F.sequence("FIRST", F.trunc(F.to_date(F.lit(date_end)), "month"),
                                                          F.expr("interval 1 month"))).alias("MONTH")))
    carry = Window.partitionBy("PATIENT_ID").orderBy("MONTH").rowsBetween(Window.unboundedPreceding, 0)
    return (months.join(per_month, ["PATIENT_ID", "MONTH"], "left")
            .withColumn("GFR", F.last("GFR", True).over(carry))
            .withColumn("UACR", F.last("UACR", True).over(carry)))

def summary(df, months):
    return {
        "patients": df.select("PATIENT_ID").distinct().count(),
        "eGFR test days": df.filter(F.col("GFR_STAGE").isNotNull()).select("PATIENT_ID", "SVC_DT").distinct().count(),
        "UACR test days": df.filter(F.col("UACR_STAGE").isNotNull()).select("PATIENT_ID", "SVC_DT").distinct().count(),
        "patient-months": months.count(),
        "patient-months with eGFR stage 1-5": months.filter(F.col("GFR").between(1, 5)).count(),
        "patient-months with eGFR stage 3-5": months.filter(F.col("GFR").between(3, 5)).count(),
        "patient-months with UACR stage": months.filter(F.col("UACR").isNotNull()).count(),
        "patient-months with UACR A2-A3": months.filter(F.col("UACR").between(2, 3)).count(),
    }

months_30, months_basket = to_patient_months(df_lab), to_patient_months(df_lab_basket)
_table = pd.DataFrame({"3.0 query": summary(df_lab, months_30), "basket query": summary(df_lab_basket, months_basket)})
_table["change"] = _table["basket query"] - _table["3.0 query"]
print("\nB) Validation HCPs")
display(_table.reset_index().rename(columns={"index": "metric"}))

# Stage per patient-month: 3.0 (rows) vs basket query (columns)
_matched = (months_30.select("PATIENT_ID", "MONTH", F.col("GFR").alias("GFR_30"), F.col("UACR").alias("UACR_30"))
            .join(months_basket.select("PATIENT_ID", "MONTH", F.col("GFR").alias("GFR_BASKET"), F.col("UACR").alias("UACR_BASKET")),
                  ["PATIENT_ID", "MONTH"], "full"))
for test in ["GFR", "UACR"]:
    print(f"{test} stage per patient-month: 3.0 (rows) vs basket query (columns)")
    display(_matched.groupBy(f"{test}_30").pivot(f"{test}_BASKET").count().orderBy(f"{test}_30"))

### 2e. Findings: 3.0 query vs basket query (run 2026-10-28, eGFR valid max = 200)

Numbers come from cell 2d. Re-run 2a to 2d after any basket change and update this summary.

**All US patients, 2025-01 to 2026-09 (patient x day x test)**

| | eGFR | UACR |
| --- | --- | --- |
| Same stage | 106.73M | 19.81M |
| Different stage | 65.3K (0.06%) | 27.4K (0.14%) |
| Only in 3.0 (lost) | 260 | 71.0K |
| Only in basket query (gained) | 0 | 0 |

**What changed and why**
* **eGFR 150-200 now stage 1 instead of stage 0** (10.4K test-days). 3.0 put these in a stage 0 that does not exist; the basket's `VALID_MAX = 200` keeps them as stage 1.
* **eGFR one-step changes** (~55K test-days, e.g. 1 -> 2, 2 -> 3, 3 -> 2). When a patient has several eGFR results on the same day, the basket query uses the preferred formula (`PRIORITY`) instead of the median of all of them, e.g. it no longer averages the Black and non-Black versions.
* **UACR mostly moves up** (2 -> 3: 18.6K, 1 -> 2: 7.4K). 3.0 mixed parts of the test (urine creatinine, albumin concentration) into the same-day median, which pulled the ratio down. The basket marks them `ROLE = PART` and never stages them.
* **Lost eGFR (260):** 212 values above 200 and 48 values of 0 (3.0 made the zeros stage 5). Both are outside the valid range.
* **Lost UACR (71.0K):** 68.4K test-days whose only "UACR" was a part of the test, not a ratio, and 2.5K with a value above 10000. Both removed on purpose.
* **No gains:** expected, because the test-name backup keeps the 3.0 rule for rows without a basket code, and the other filters (specialty, dates, value format) are still as in 3.0.

**Validation HCPs (10 HCPs, after the HCP universe join)**
* Same 225 patients, 878 eGFR and 289 UACR test-days, and 6,139 patient-months as 3.0 (stage carried forward month by month until the next test).
* The only change: 26 patient-months move from eGFR stage 0 to stage 1 (the patient with an eGFR between 150 and 200).

**Conclusion:** the basket query keeps all 3.0 data, except values that are not valid results, and the stage changes come from the two corrections above. Nothing downstream uses it yet; section 2 still uses `df_lab` from the 3.0 cell.

## Step 3: Config changes (items 7-12)

The run settings that were typed into the 3.0 SQL now come from one settings block (`LAB_CONFIG` in 3a): dates, specialty filter, date formats, source table and column names. Changing a setting is now a one-line edit, with no SQL change. The basket rules from Step 2 are kept as they are.

The specialty list keeps the 3.0 value for now, so the comparison shows only the effect of the new structure. Changing it (e.g. adding cardiology, endocrinology or nephrology for HF / T1D) is a separate decision.

| Cell | What it does |
| --- | --- |
| 3a | The lab run settings, and checks that the source tables and columns exist |
| 3b | **The lab query** (built from the basket and the settings) |
| 3c | Runs the query |
| 3d | Compares the Step 2 query with the Step 3 query (the evidence for 3e) |
| 3e | Findings: what changed compared with Step 2 |

In [0]:
# 3a. Lab run settings (items 7-12). Edit these values instead of the SQL.
#     For now they live here; once approved they move to 00_cohort_config.
LAB_CONFIG = {
    # item 8: one start date (3.0 had '2021-11-30' in the SQL plus DATA_START_DATE in Spark)
    "start_date": DATA_START_DATE,
    # item 9: end date inside the SQL (3.0 pulled all history and cut it in Spark)
    "end_date": date_end,
    # item 7: prescriber specialties kept. Same as 3.0 for now; None = no specialty filter
    "specialties": ["FM", "FPG", "GP", "GPM", "IFP", "IM", "IMG", "IPM", "NRP", "PHA"],
    # item 10: service date formats, tried in this order. Rows that match none are counted in 3d
    "date_formats": ["MM/DD/YYYY", "DDMONYYYY"],
    # items 11-12: source table and its column names
    "source": {
        "lab_name": "Quest",
        "table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_QUEST_TO_LAAD_HIST",
        "columns": {"patient_id": "PATIENT_ID", "npi": "NPI_ID", "service_date": "SVC_DT", "loinc": "LOINC_CD",
                    "test_name": "LCL_LAB_TEST_NM_CLEAN", "unit": "LCL_LAB_TEST_RSLT_UOM_CD_CLEAN",
                    "value": "LCL_LAB_TEST_VAL_TXT"},
    },
    "provider": {
        "table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR",
        "columns": {"npi": "NPI_NUMBER", "specialty": "PRIMARY_SPECIALTY_CODE"},
    },
}

# ---------------------------------------------------------------------------
# Checks: the cell stops with a clear message if a setting is wrong
# ---------------------------------------------------------------------------
assert LAB_CONFIG["start_date"] <= str(LAB_CONFIG["end_date"]), "start_date is after end_date"
assert LAB_CONFIG["date_formats"], "date_formats is empty"
assert LAB_CONFIG["specialties"] is None or len(LAB_CONFIG["specialties"]) > 0, "specialties: use None for no filter"

for _name in ["source", "provider"]:
    _t = LAB_CONFIG[_name]
    try:
        Get_Data_Snowflakes(f"SELECT {', '.join(_t['columns'].values())} FROM {_t['table']} LIMIT 1").collect()
    except Exception as e:
        raise ValueError(f"{_name}: table {_t['table']} or one of its columns {list(_t['columns'].values())} "
                         f"was not found. Check LAB_CONFIG.") from e

print("Settings OK: source and provider tables and all columns found")
display(pd.DataFrame([
    ("start date", LAB_CONFIG["start_date"]),
    ("end date", str(LAB_CONFIG["end_date"])),
    ("specialties", "no filter" if LAB_CONFIG["specialties"] is None else ", ".join(LAB_CONFIG["specialties"])),
    ("date formats", ", ".join(LAB_CONFIG["date_formats"])),
    ("lab source", f"{LAB_CONFIG['source']['lab_name']}: {LAB_CONFIG['source']['table']}"),
    ("provider table", LAB_CONFIG["provider"]["table"]),
], columns=["setting", "value"]))

In [0]:
# 3b. The lab query, built from the basket (2a) and the run settings (3a).
#     Same logic as the Step 2 query; the only differences are that dates, specialties,
#     date formats, table names and column names now come from LAB_CONFIG.
def build_lab_query(basket, rules, cfg):
    c, p = cfg["source"]["columns"], cfg["provider"]["columns"]
    codes = basket[basket["MATCH_TYPE"] == "LOINC"]
    names = basket[basket["MATCH_TYPE"] == "NAME"]
    # name backups: if several patterns match, the most specific (longest) one wins
    names = names.assign(_len=names["MATCH_VALUE"].str.len()).sort_values("_len", ascending=False)
    code_values = ",\n        ".join(
        f"('{r.MATCH_VALUE}', '{r.TEST_TYPE}', '{r.ROLE}', {r.PRIORITY or 'NULL'})" for r in codes.itertuples())
    name_type = " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN '{r.TEST_TYPE}'" for r in names.itertuples())
    name_role = " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN '{r.ROLE}'" for r in names.itertuples())
    name_prio = " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN {r.PRIORITY or 'NULL'}" for r in names.itertuples())
    valid = "\n           OR ".join(
        f"(TEST_TYPE = '{t}' AND UOM IN ({_q(r['units'])}) AND TEST_NUM BETWEEN {r['valid_min']} AND {r['valid_max']})"
        for t, r in rules.items())
    stages = ",\n  ".join(
        f"CASE WHEN TEST_TYPE = '{t}' THEN CASE "
        + " ".join(f"WHEN MEDIAN_TEST_RESULT < {cut} THEN {s}" for cut, s in r["cuts"])
        + f" ELSE {r['top_stage']} END END AS {t}_STAGE"
        for t, r in rules.items())
    # item 10: try each date format in order; no match gives NULL (counted in 3d) instead of an error
    svc_date = "COALESCE(" + ", ".join(
        f"TRY_TO_DATE(LEFT({c['service_date']}, {len(f)}), '{f}')" for f in cfg["date_formats"]) + ")"
    # item 7: specialty filter only when a list is given
    specialty_join = (f"JOIN {cfg['provider']['table']} dct ON t.NPI_ID = dct.{p['npi']}\n  "
                      f"  AND dct.{p['specialty']} IN ({_q(cfg['specialties'])})") if cfg["specialties"] else ""
    return f"""
WITH basket AS (
  SELECT * FROM VALUES
        {code_values}
  AS b(LOINC_CD, TEST_TYPE, ROLE, PRIORITY)
),
src AS (   -- same row filters as 3.0 (value format and patient ID rules are items 6 and 13, a later step)
  SELECT CAST(CAST({c['patient_id']} AS INT) AS VARCHAR(64)) AS PATIENT_ID,
         {c['npi']} AS NPI_ID,
         {svc_date} AS SVC_DT_STD,
         {c['loinc']} AS LOINC_CD,
         UPPER({c['test_name']}) AS TEST_NM,
         UPPER(REPLACE({c['unit']}, ' ', '')) AS UOM,
         TRY_TO_DOUBLE(TRIM({c['value']})) AS TEST_NUM
  FROM {cfg['source']['table']}
  WHERE REGEXP_LIKE({c['patient_id']}, '^[0-9]+$')
    AND {c['value']} IS NOT NULL
    AND TRIM({c['value']}) NOT ILIKE '%-%'
    AND TRIM({c['value']}) NOT ILIKE '%>%'
    AND TRIM({c['value']}) NOT ILIKE '%<%'
    AND REGEXP_LIKE(TRIM({c['value']}), '^[0-9]+(\\.[0-9]+)?$')
    AND ({c['loinc']} IN (SELECT LOINC_CD FROM basket) OR UPPER({c['test_name']}) LIKE ANY ({_q(names['MATCH_VALUE'])}))
),
typed AS (   -- basket first; test name only as backup for codes not in the basket
  SELECT s.PATIENT_ID, s.NPI_ID, s.SVC_DT_STD, s.LOINC_CD, s.UOM, s.TEST_NUM,
         COALESCE(b.TEST_TYPE, CASE {name_type} END) AS TEST_TYPE,
         COALESCE(b.ROLE, CASE {name_role} END) AS ROLE,
         IFF(b.LOINC_CD IS NULL, 'NAME', 'BASKET') AS MATCHED_BY,
         COALESCE(b.PRIORITY, CASE {name_prio} END) AS PRIORITY
  FROM src s LEFT JOIN basket b ON s.LOINC_CD = b.LOINC_CD
),
kept AS (    -- dates and specialties from the settings; units and valid range per test from the basket
  SELECT t.*
  FROM typed t
  {specialty_join}
  WHERE t.ROLE = 'RESULT'
    AND ({valid})
    AND t.SVC_DT_STD BETWEEN '{cfg['start_date']}' AND '{cfg['end_date']}'
    AND t.NPI_ID IS NOT NULL
  QUALIFY PRIORITY = MIN(PRIORITY) OVER (PARTITION BY t.PATIENT_ID, t.NPI_ID, t.SVC_DT_STD, t.TEST_TYPE)
),
vals AS (    -- same DISTINCT as 3.0 before the median
  SELECT DISTINCT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE, TEST_NUM FROM kept
),
info AS (
  SELECT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE,
         LISTAGG(DISTINCT COALESCE(LOINC_CD, 'NO_LOINC'), ',') AS LOINC_CDS,
         MIN(MATCHED_BY) AS MATCHED_BY
  FROM kept GROUP BY 1, 2, 3, 4
),
agg AS (
  SELECT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE, MEDIAN(TEST_NUM) AS MEDIAN_TEST_RESULT
  FROM vals GROUP BY 1, 2, 3, 4
)
SELECT a.PATIENT_ID, a.SVC_DT_STD AS DATE, a.NPI_ID AS NPI_NUMBER, a.TEST_TYPE, a.MEDIAN_TEST_RESULT,
  {stages},
  i.LOINC_CDS, i.MATCHED_BY
FROM agg a JOIN info i USING (PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE)
"""

lab_query_config = build_lab_query(lab_basket, lab_test_rules, LAB_CONFIG)
print(lab_query_config)

In [0]:
# 3c. Run the lab query (3b). Dates are already applied in the SQL, so only the HCP universe
#     and the validation filter are applied here.
df_lab_config_all = (Get_Data_Snowflakes(lab_query_config)
    .join(_hcp_universe, how="inner", on="NPI_NUMBER")
    .withColumnRenamed("DATE", "SVC_DT")
    .withColumn("GFR_STAGE", col("GFR_STAGE").cast("int"))
    .withColumn("UACR_STAGE", col("UACR_STAGE").cast("int")))
df_lab_config_all, test_patients_config = apply_hcp_filter(df_lab_config_all)

# Same 4 columns as df_lab, so section 2 can use it unchanged once approved
df_lab_config = df_lab_config_all.select("PATIENT_ID", "SVC_DT", "GFR_STAGE", "UACR_STAGE")

display(df_lab_config_all.orderBy("PATIENT_ID", "SVC_DT"))

In [0]:
# 3d. Compare the Step 2 query (2b) with the Step 3 query (3b).
#     Part A: all US patients, result by result (same patient, day, NPI, test)
#     Part B: the 10 validation HCPs, month by month (uses to_patient_months and summary from 2d)
#     Part C: how much data each query pulls from Snowflake
#     Part D: source rows lost because the service date could not be read

# ---------------------------------------------------------------------------
# Part A: all US patients, both queries side by side in Snowflake
# ---------------------------------------------------------------------------
_compare_sql = f"""
WITH s2 AS (SELECT * FROM ({lab_query_basket}) WHERE DATE BETWEEN '{CMP_START}' AND '{date_end}'),
     s3 AS (SELECT * FROM ({lab_query_config}) WHERE DATE BETWEEN '{CMP_START}' AND '{date_end}')
SELECT COALESCE(s2.TEST_TYPE, s3.TEST_TYPE) AS TEST_TYPE,
  CASE WHEN s2.PATIENT_ID IS NULL THEN '3. only Step 3 query (gained)'
       WHEN s3.PATIENT_ID IS NULL THEN '4. only Step 2 query (lost)'
       WHEN EQUAL_NULL(s2.GFR_STAGE, s3.GFR_STAGE) AND EQUAL_NULL(s2.UACR_STAGE, s3.UACR_STAGE)
            AND s2.MEDIAN_TEST_RESULT = s3.MEDIAN_TEST_RESULT THEN '1. identical'
       ELSE '2. different value or stage' END AS RESULT,
  COUNT(*) AS TEST_DAYS
FROM s2 FULL OUTER JOIN s3 USING (PATIENT_ID, DATE, NPI_NUMBER, TEST_TYPE)
GROUP BY 1, 2
"""
print(f"A) All US patients, {CMP_START} to {date_end}: patient x day x NPI x test")
display(Get_Data_Snowflakes(_compare_sql).orderBy("TEST_TYPE", "RESULT"))

# ---------------------------------------------------------------------------
# Part B: validation HCPs, month by month
# ---------------------------------------------------------------------------
months_config = to_patient_months(df_lab_config)
_table = pd.DataFrame({"Step 2 query": summary(df_lab_basket, months_basket),
                       "Step 3 query": summary(df_lab_config, months_config)})
_table["change"] = _table["Step 3 query"] - _table["Step 2 query"]
print("\nB) Validation HCPs")
display(_table.reset_index().rename(columns={"index": "metric"}))

# ---------------------------------------------------------------------------
# Part C: data pulled from Snowflake by each query (all US, no date cut in Spark)
# ---------------------------------------------------------------------------
_size = []
for _label, _q_sql in [("Step 2 query", lab_query_basket), ("Step 3 query", lab_query_config)]:
    _r = Get_Data_Snowflakes(f"SELECT COUNT(*) AS ROWS_PULLED, MIN(DATE) AS FIRST_DATE, MAX(DATE) AS LAST_DATE FROM ({_q_sql})").collect()[0]
    _size.append((_label, int(_r["ROWS_PULLED"]), str(_r["FIRST_DATE"]), str(_r["LAST_DATE"])))
print("\nC) Data pulled from Snowflake (all US)")
display(pd.DataFrame(_size, columns=["query", "rows pulled", "first date", "last date"]))

# ---------------------------------------------------------------------------
# Part D: source rows (basket tests) whose service date matches none of the date formats
# ---------------------------------------------------------------------------
_src, _c = LAB_CONFIG["source"], LAB_CONFIG["source"]["columns"]
_svc = "COALESCE(" + ", ".join(f"TRY_TO_DATE(LEFT({_c['service_date']}, {len(f)}), '{f}')" for f in LAB_CONFIG["date_formats"]) + ")"
_codes_list = _q(lab_basket[lab_basket["MATCH_TYPE"] == "LOINC"]["MATCH_VALUE"])
_names_list = _q(lab_basket[lab_basket["MATCH_TYPE"] == "NAME"]["MATCH_VALUE"])
_dates_sql = f"""
SELECT COUNT(*) AS SOURCE_ROWS,
       COUNT_IF({_c['service_date']} IS NULL) AS DATE_EMPTY,
       COUNT_IF({_c['service_date']} IS NOT NULL AND {_svc} IS NULL) AS DATE_IN_UNKNOWN_FORMAT,
       ARRAY_SLICE(ARRAY_AGG(DISTINCT IFF({_c['service_date']} IS NOT NULL AND {_svc} IS NULL, {_c['service_date']}, NULL)), 0, 5) AS UNKNOWN_FORMAT_EXAMPLES
FROM {_src['table']}
WHERE {_c['loinc']} IN ({_codes_list}) OR UPPER({_c['test_name']}) LIKE ANY ({_names_list})
"""
print("\nD) Source rows for the basket tests whose service date could not be read")
display(Get_Data_Snowflakes(_dates_sql))

### 3e. Findings: Step 2 query vs Step 3 query (run 2026-10-28)

Numbers come from cell 3d. Re-run 3a to 3d after changing a setting and update this summary.

**Result: same output, less data pulled.** Moving the settings out of the SQL changed nothing in the results.

| Check | Step 2 query | Step 3 query | Change |
| --- | --- | --- | --- |
| All US eGFR results, 2025-01 to 2026-09 (patient x day x NPI) | 107.17M | 107.17M | all identical (same value and stage) |
| All US UACR results, same period | 19.86M | 19.86M | all identical |
| Validation HCPs: patients / eGFR test days / UACR test days | 225 / 878 / 289 | 225 / 878 / 289 | none |
| Validation HCPs: patient-months (and by stage) | 6,139 | 6,139 | none |
| Rows pulled from Snowflake (all US) | 312.9M (2021-11-30 to 2026-10-05) | 252.2M (2023-01-01 to 2026-09-30) | **-60.7M rows (-19%)** |

**What changed and why**
* **Dates (items 8-9):** the query now reads `DATA_START_DATE` and `date_end` directly. Before, it pulled from 2021-11-30 and past the end date, and Spark threw that data away afterwards. One start date instead of two, and 19% less data moved out of Snowflake.
* **Date formats (item 10):** a date in an unknown format no longer breaks the query; it is skipped and counted. Today there are none: of 1.72B source rows for the basket tests, 38 have an empty service date (3.0 skipped those too) and 0 have an unknown format.
* **Specialties (item 7):** now one setting. Still the 3.0 list of primary care specialties, so nothing changed. Setting it to `None` removes the filter.
* **Tables and columns (items 11-12):** names now sit in `LAB_CONFIG`, and 3a checks they exist before the query runs. A renamed table or column now gives a clear message instead of a SQL error.

**Open decision:** the specialty list. The earlier national check showed that removing the primary-care-only filter adds about 16% more eGFR patients and 13% more UACR patients, with no losses. For HF and T1D, cardiology, endocrinology and nephrology could matter.

Nothing downstream uses this query yet; section 2 still uses `df_lab` from the 3.0 cell.

## Step 4: Code changes (items 6, 13 and 15) and a report of dropped rows

The last hardcoded parts of the query become shared rules, written once (in 4a):
* **Value format (item 6):** lab flags like `^` are removed (`75^` -> 75), a leading dot is read (`.5` -> 0.5), and "above / below" results (`>120`, `<4`) are kept **when every value in that range has the same stage**. `>60` is not staged, because it could be stage 1 or 2.
* **Patient ID (item 13):** same 3.0 rule (digits only), now in one place, and the rows it drops are counted.
* **Output columns (item 15):** the stage columns follow the tests in the basket, so switching a test to `ACTIVE = Y` adds its column with no code change.
* **Dropped-rows report:** every source row is counted under the first reason it is dropped (patient ID, date, unit, value, range, specialty...), so data loss shows up on every run.

Basket (Step 2) and settings (Step 3) are kept as they are.

| Cell | What it does |
| --- | --- |
| 4a | Value and patient ID rules, and the per-test rules read from the basket |
| 4b | **The lab query** (basket + settings + rules), and the dropped-rows report built from the same query |
| 4c | Runs the query |
| 4d | Compares the Step 3 query with the Step 4 query, and shows the dropped-rows report (the evidence for 4e) |
| 4e | Findings: what changed compared with Step 3 |

In [0]:
# 4a. Shared rules (items 6, 13, 15). Edit these instead of the SQL.
import builtins   # min() / max() are replaced by the Spark versions in this notebook

# item 6: how a result value is read
VALUE_RULES = {
    "flag_chars": ["^"],               # lab flags removed before reading the number ('75^' -> 75)
    "censored_prefixes": [">", "<"],   # '>120', '< 3': the value is only known to be above / below
    "stage_censored_if_clear": True,   # stage '>X' / '<X' only when every value in that range has the same stage
}
# item 13: which patient IDs are kept (3.0 rule: digits only, leading zeros removed)
PATIENT_ID_RULE = {"pattern": "^[0-9]+$"}

def sql_value(col):
    """SQL for a result value: (qualifier, number). Qualifier is '=', '>' or '<'; number is NULL if not a number."""
    clean = f"TRIM(TRANSLATE({col}, '{''.join(VALUE_RULES['flag_chars'])}', ''))"
    prefixes = VALUE_RULES["censored_prefixes"]
    qual = (f"IFF(LEFT({clean}, 1) IN ({_q(prefixes)}), LEFT({clean}, 1), '=')" if prefixes else "'='")
    num_txt = f"TRIM(LTRIM({clean}, '{''.join(prefixes)}='))" if prefixes else clean
    num = f"IFF(REGEXP_LIKE({num_txt}, '^[0-9]*[.]?[0-9]+$'), TRY_TO_DOUBLE({num_txt}), NULL)"
    return qual, num

def sql_patient_id(col):
    """SQL for the patient ID: (is valid, cleaned ID)."""
    ok = f"REGEXP_LIKE({col}, '{PATIENT_ID_RULE['pattern']}')"
    return ok, f"IFF({ok}, CAST(CAST({col} AS NUMBER(38, 0)) AS VARCHAR(64)), NULL)"

# item 15: one set of rules per active test, read from the basket (2a). A test without cut-offs gets a value but no stage.
lab_rules_step4 = {}
for t, r in lab_basket[lab_basket["ROLE"] == "RESULT"].groupby("TEST_TYPE").first().iterrows():
    pairs = [p.split(":") for p in r["STAGE_CUTOFFS"].split("|")] if r["STAGE_CUTOFFS"] else []
    cuts = [(float(v), int(s)) for v, s in pairs if v != "else"]
    assert [c for c, _ in cuts] == sorted(c for c, _ in cuts), f"{t}: STAGE_CUTOFFS must go from low to high"
    lab_rules_step4[t] = {"units": r["UNITS"].split("|"), "valid_min": r["VALID_MIN"], "valid_max": r["VALID_MAX"],
                          "cuts": cuts, "top_stage": next((int(s) for v, s in pairs if v == "else"), None)}

# Checks
assert not any(ch.isdigit() or ch == "." for ch in VALUE_RULES["flag_chars"]), "flag_chars cannot contain digits or '.'"
assert all(r["top_stage"] is not None for r in lab_rules_step4.values() if r["cuts"]), "STAGE_CUTOFFS needs an 'else' stage"

print("Rules OK")
display(pd.DataFrame([
    (t, ", ".join(r["units"]), f"{r['valid_min']} - {r['valid_max']}",
     ", ".join(f"<{c:g}: {s}" for c, s in r["cuts"]) + (f", else: {r['top_stage']}" if r["cuts"] else "no stage"),
     (f">{builtins.max(c for c, _ in r['cuts']):g} and <{builtins.min(c for c, _ in r['cuts']):g}" if r["cuts"] and VALUE_RULES["stage_censored_if_clear"] else "none"))
    for t, r in lab_rules_step4.items()],
    columns=["test", "units", "valid range", "stages", "above / below values staged (at or beyond)"]))

In [0]:
# 4b. The lab query, built from the basket (2a), the settings (3a) and the rules (4a).
#     Same logic as the Step 3 query, plus: shared value and patient ID rules, output columns from the basket,
#     and a dropped-rows report that reuses the same steps so both always agree.
def build_lab_query_step4(basket, rules, cfg):
    c, p = cfg["source"]["columns"], cfg["provider"]["columns"]
    codes = basket[basket["MATCH_TYPE"] == "LOINC"]
    names = basket[basket["MATCH_TYPE"] == "NAME"]
    names = names.assign(_len=names["MATCH_VALUE"].str.len()).sort_values("_len", ascending=False)   # longest pattern wins
    code_values = ",\n        ".join(
        f"('{r.MATCH_VALUE}', '{r.TEST_TYPE}', '{r.ROLE}', {r.PRIORITY or 'NULL'})" for r in codes.itertuples())
    name_case = lambda field: " ".join(f"WHEN s.TEST_NM LIKE '{r.MATCH_VALUE}' THEN {field(r)}" for r in names.itertuples())
    svc_date = "COALESCE(" + ", ".join(
        f"TRY_TO_DATE(LEFT({c['service_date']}, {len(f)}), '{f}')" for f in cfg["date_formats"]) + ")"
    pid_ok, pid = sql_patient_id("s.PID_RAW")
    qual, num = sql_value("s.VAL_RAW")

    def per_test(rule):   # one CASE per test, generated from the rules
        return "COALESCE(CASE TEST_TYPE " + " ".join(f"WHEN '{t}' THEN {rule(r)}" for t, r in rules.items()) + " END, FALSE)"
    def clear_stage(r):   # '>X' / '<X' only when the whole range has one stage
        if not (r["cuts"] and VALUE_RULES["stage_censored_if_clear"]):
            return "QUAL = '='"
        return (f"(QUAL = '=' OR (QUAL = '>' AND TEST_NUM >= {builtins.max(x for x, _ in r['cuts'])})"
                f" OR (QUAL = '<' AND TEST_NUM <= {builtins.min(x for x, _ in r['cuts'])}))")
    unit_ok = per_test(lambda r: f"UOM IN ({_q(r['units'])})")
    range_ok = per_test(lambda r: f"TEST_NUM BETWEEN {r['valid_min']} AND {r['valid_max']}")
    stage_ok = per_test(clear_stage)
    stages = ",\n  ".join(
        f"CASE WHEN TEST_TYPE = '{t}' THEN CASE "
        + " ".join(f"WHEN MEDIAN_TEST_RESULT < {x:g} THEN {s}" for x, s in r["cuts"])
        + f" ELSE {r['top_stage']} END END AS {t}_STAGE"
        for t, r in rules.items() if r["cuts"])
    if cfg["specialties"]:
        spec_cte = (f",\nspec AS (SELECT DISTINCT {p['npi']} AS NPI_ID FROM {cfg['provider']['table']}"
                    f" WHERE {p['specialty']} IN ({_q(cfg['specialties'])}))")
        spec_join, spec_ok = "LEFT JOIN spec sp ON t.NPI_ID = sp.NPI_ID", "sp.NPI_ID IS NOT NULL"
    else:
        spec_cte, spec_join, spec_ok = "", "", "TRUE"

    steps = f"""
WITH basket AS (
  SELECT * FROM VALUES
        {code_values}
  AS b(LOINC_CD, TEST_TYPE, ROLE, PRIORITY)
){spec_cte},
src AS (     -- source rows for the basket tests (by code, or by name as backup)
  SELECT {c['patient_id']} AS PID_RAW, {c['npi']} AS NPI_ID, {svc_date} AS SVC_DT_STD, {c['loinc']} AS LOINC_CD,
         UPPER({c['test_name']}) AS TEST_NM, UPPER(REPLACE({c['unit']}, ' ', '')) AS UOM, TRIM({c['value']}) AS VAL_RAW
  FROM {cfg['source']['table']}
  WHERE {c['loinc']} IN (SELECT LOINC_CD FROM basket) OR UPPER({c['test_name']}) LIKE ANY ({_q(names['MATCH_VALUE'])})
),
parsed AS (  -- shared rules: patient ID (item 13), value (item 6), test from the basket
  SELECT s.*, {pid_ok} AS PID_OK, {pid} AS PATIENT_ID,
         {qual} AS QUAL, {num} AS TEST_NUM,
         COALESCE(b.TEST_TYPE, CASE {name_case(lambda r: repr(r.TEST_TYPE))} END) AS TEST_TYPE,
         COALESCE(b.ROLE, CASE {name_case(lambda r: repr(r.ROLE))} END) AS ROLE,
         IFF(b.LOINC_CD IS NULL, 'NAME', 'BASKET') AS MATCHED_BY,
         COALESCE(b.PRIORITY, CASE {name_case(lambda r: r.PRIORITY or 'NULL')} END) AS PRIORITY
  FROM src s LEFT JOIN basket b ON s.LOINC_CD = b.LOINC_CD
  WHERE s.SVC_DT_STD IS NULL OR s.SVC_DT_STD BETWEEN '{cfg['start_date']}' AND '{cfg['end_date']}'
),
checks AS (  -- one TRUE/FALSE per rule, used by the query and by the dropped-rows report
  SELECT t.*, {unit_ok} AS UNIT_OK, {range_ok} AS RANGE_OK, {stage_ok} AS STAGE_OK, {spec_ok} AS SPEC_OK
  FROM parsed t {spec_join}
)"""

    query = steps + f""",
kept AS (
  SELECT * FROM checks
  WHERE PID_OK AND SVC_DT_STD IS NOT NULL AND NPI_ID IS NOT NULL AND ROLE = 'RESULT' AND TEST_NUM IS NOT NULL
    AND UNIT_OK AND RANGE_OK AND STAGE_OK AND SPEC_OK
  -- same day: best priority first, exact values before above / below values
  QUALIFY PRIORITY * 10 + IFF(QUAL = '=', 0, 1)
          = MIN(PRIORITY * 10 + IFF(QUAL = '=', 0, 1)) OVER (PARTITION BY PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE)
),
vals AS (    -- '<X' is staged as just below X
  SELECT DISTINCT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE, IFF(QUAL = '<', TEST_NUM - 0.0001, TEST_NUM) AS TEST_NUM FROM kept
),
info AS (
  SELECT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE,
         LISTAGG(DISTINCT COALESCE(LOINC_CD, 'NO_LOINC'), ',') AS LOINC_CDS,
         MIN(MATCHED_BY) AS MATCHED_BY,
         MAX(IFF(QUAL = '=', NULL, VAL_RAW)) AS ABOVE_BELOW_VALUE
  FROM kept GROUP BY 1, 2, 3, 4
),
agg AS (
  SELECT PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE, MEDIAN(TEST_NUM) AS MEDIAN_TEST_RESULT
  FROM vals GROUP BY 1, 2, 3, 4
)
SELECT a.PATIENT_ID, a.SVC_DT_STD AS DATE, a.NPI_ID AS NPI_NUMBER, a.TEST_TYPE, a.MEDIAN_TEST_RESULT,
  {stages},
  i.LOINC_CDS, i.MATCHED_BY, i.ABOVE_BELOW_VALUE
FROM agg a JOIN info i USING (PATIENT_ID, NPI_ID, SVC_DT_STD, TEST_TYPE)
"""

    report = steps + """
SELECT TEST_TYPE,
  CASE WHEN NOT PID_OK                     THEN '01. patient ID not numeric'
       WHEN SVC_DT_STD IS NULL             THEN '02. service date empty or unknown format'
       WHEN NPI_ID IS NULL                 THEN '03. NPI missing'
       WHEN ROLE = 'PART'                  THEN '04. part of a test, not a result'
       WHEN COALESCE(VAL_RAW, '') = ''     THEN '05. value empty'
       WHEN TEST_NUM IS NULL               THEN '06. value is text, not a number'
       WHEN NOT UNIT_OK                    THEN '07. unit not allowed'
       WHEN NOT RANGE_OK                   THEN '08. outside valid range'
       WHEN NOT STAGE_OK                   THEN '09. above / below value, stage unclear'
       WHEN NOT SPEC_OK                    THEN '10. specialty not in list'
       ELSE '11. kept' END AS REASON,
  COUNT(*) AS SOURCE_ROWS, ANY_VALUE(VAL_RAW) AS EXAMPLE_VALUE
FROM checks GROUP BY 1, 2
"""
    return query, report

lab_query_step4, lab_drop_report_step4 = build_lab_query_step4(lab_basket, lab_rules_step4, LAB_CONFIG)
print(lab_query_step4)

In [0]:
# 4c. Run the lab query (4b). Stage columns follow the tests in the basket (item 15).
_stage_cols = [f"{t}_STAGE" for t, r in lab_rules_step4.items() if r["cuts"]]

df_lab_step4_all = (Get_Data_Snowflakes(lab_query_step4)
    .join(_hcp_universe, how="inner", on="NPI_NUMBER")
    .withColumnRenamed("DATE", "SVC_DT")
    .withColumns({c_: col(c_).cast("int") for c_ in _stage_cols}))
df_lab_step4_all, test_patients_step4 = apply_hcp_filter(df_lab_step4_all)

# Patient, date and one stage column per test, as section 2 expects
df_lab_step4 = df_lab_step4_all.select("PATIENT_ID", "SVC_DT", *_stage_cols)
print(f"Output columns: {df_lab_step4.columns}")

display(df_lab_step4_all.orderBy("PATIENT_ID", "SVC_DT"))

In [0]:
# 4d. Compare the Step 3 query (3b) with the Step 4 query (4b).
#     Part A: all US patients, result by result (same patient, day, NPI, test)
#     Part B: the 10 validation HCPs, month by month (uses to_patient_months and summary from 2d)
#     Part C: dropped-rows report, all US source rows for the basket tests in the date range

# ---------------------------------------------------------------------------
# Part A: all US patients, both queries side by side in Snowflake
# ---------------------------------------------------------------------------
_compare_sql = f"""
WITH s3 AS (SELECT * FROM ({lab_query_config}) WHERE DATE BETWEEN '{CMP_START}' AND '{date_end}'),
     s4 AS (SELECT * FROM ({lab_query_step4}) WHERE DATE BETWEEN '{CMP_START}' AND '{date_end}')
SELECT COALESCE(s3.TEST_TYPE, s4.TEST_TYPE) AS TEST_TYPE,
  CASE WHEN s3.PATIENT_ID IS NULL THEN '3. only Step 4 query (gained)'
       WHEN s4.PATIENT_ID IS NULL THEN '4. only Step 3 query (lost)'
       WHEN EQUAL_NULL(s3.GFR_STAGE, s4.GFR_STAGE) AND EQUAL_NULL(s3.UACR_STAGE, s4.UACR_STAGE)
            AND s3.MEDIAN_TEST_RESULT = s4.MEDIAN_TEST_RESULT THEN '1. identical'
       WHEN EQUAL_NULL(s3.GFR_STAGE, s4.GFR_STAGE) AND EQUAL_NULL(s3.UACR_STAGE, s4.UACR_STAGE) THEN '2a. same stage, value changed'
       ELSE '2b. different stage' END AS RESULT,
  CASE WHEN RESULT LIKE '2b%' THEN 'stage ' || COALESCE(s3.GFR_STAGE, s3.UACR_STAGE) || ' -> ' || COALESCE(s4.GFR_STAGE, s4.UACR_STAGE)
       WHEN RESULT LIKE '3%' AND s4.ABOVE_BELOW_VALUE IS NOT NULL THEN 'above / below value with a clear stage (e.g. >120, <4)'
       WHEN RESULT LIKE '3%' THEN 'value with a lab flag or a leading dot (e.g. 75^, .5)'
  END AS DETAIL,
  COUNT(*) AS TEST_DAYS, COUNT(DISTINCT COALESCE(s3.PATIENT_ID, s4.PATIENT_ID)) AS PATIENTS
FROM s3 FULL OUTER JOIN s4 USING (PATIENT_ID, DATE, NPI_NUMBER, TEST_TYPE)
GROUP BY 1, 2, 3
"""
print(f"A) All US patients, {CMP_START} to {date_end}: patient x day x NPI x test")
display(Get_Data_Snowflakes(_compare_sql).orderBy("TEST_TYPE", "RESULT", F.desc("TEST_DAYS")))

# ---------------------------------------------------------------------------
# Part B: validation HCPs, month by month
# ---------------------------------------------------------------------------
months_step4 = to_patient_months(df_lab_step4)
_table = pd.DataFrame({"Step 3 query": summary(df_lab_config, months_config),
                       "Step 4 query": summary(df_lab_step4, months_step4)})
_table["change"] = _table["Step 4 query"] - _table["Step 3 query"]
print("\nB) Validation HCPs")
display(_table.reset_index().rename(columns={"index": "metric"}))

# ---------------------------------------------------------------------------
# Part C: dropped-rows report (each source row counted under the first reason it is dropped)
# ---------------------------------------------------------------------------
print(f"\nC) Source rows for the basket tests, {LAB_CONFIG['start_date']} to {LAB_CONFIG['end_date']}, by reason")
display(Get_Data_Snowflakes(lab_drop_report_step4).orderBy("TEST_TYPE", "REASON"))

### 4e. Findings: Step 3 query vs Step 4 query (run 2026-10-28)

Numbers come from cell 4d (Part A for 2025-01 to 2026-09; the 2023-2024 line is the same comparison run once for the earlier dates). Re-run 4a to 4d after changing a rule and update this summary.

**Result: nothing lost, a little gained, and every dropped row now has a reason.**

| All US patients (patient x day x NPI x test) | eGFR | UACR |
| --- | --- | --- |
| Identical to Step 3, 2025-01 to 2026-09 | 107.17M | 19.86M |
| Gained 2025+: above / below value with a clear stage (`>90`, `>120`, `<4`) | +35.7K (26.8K patients) | +535 |
| Gained 2025+: value with a lab flag or leading dot (`75^`, `.5`) | 0 | +747 |
| Gained 2023-2024: value with a lab flag (`75^`) | +75.7K (71.0K patients) | +10.1K |
| Gained 2023-2024: above / below value with a clear stage | +568 | +568 |
| Lost | 0 | 0 |
| Different stage | 3 (2023-2024) | 0 |

Validation HCPs: no change (same 225 patients, test days and 6,139 patient-months).

**What changed and why**
* **Value format (item 6):** lab flags (`^`) only appear in 2023 data, so they add history but nothing after 2025. Above / below values are staged only when the stage is certain: `>90` is always stage 1, `<15` always stage 5, `<30` always A1. On the same day an exact value always wins over an above / below value.
* **Patient ID (item 13):** nothing to recover. All rows dropped by this rule have an empty patient ID or `-1` (unknown patient).
* **Output columns (item 15):** stage columns now come from the basket (today `GFR_STAGE`, `UACR_STAGE`, as before).

**Dropped-rows report (all US source rows for the basket tests, 2023-01 to 2026-09)**

| Reason (first one that applies) | eGFR rows | UACR rows |
| --- | --- | --- |
| Patient ID empty or `-1` | 65.6M | 11.8M |
| Service date empty | 27 | 11 |
| NPI missing | 24.3M | 3.8M |
| Part of a test, not a result | - | 16.0M |
| Value empty | 1.6M | 0.8M |
| Value is text (`DNR`, `NOTE`, `See Comment`...) | 10.8M | 6.0M |
| Unit not allowed | 0 | 32.8K |
| Outside valid range | 4.6M | 26.2K |
| Above / below value, stage unclear (`>60`) | 827K | 314 |
| Specialty not in the list | **211.4M** | **21.2M** |
| **Kept** | **631.9M** | **106.9M** |

* The specialty filter is by far the biggest cut (about 1 in 4 eGFR rows). This feeds the specialty decision in the final summary.
* `>60` is the only real data left out by the value rules (827K eGFR rows). It means "eGFR is 60 or more", so stage 1 or 2, not CKD stage 3+. **New decision for the final summary:** keep it out (as today), or use it as "no CKD stage 3+".

Nothing downstream uses this query yet; section 2 still uses `df_lab` from the 3.0 cell.

## 2. Data cleanup and processing

In [0]:
# Cohort-aware month-grain cleanup with forward-fill

import sys
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col

# Find the end date limit
end_date_limit = str(df_lab.agg({"SVC_DT": "max"}).collect()[0]['max(SVC_DT)'])

# Define forward fill windows (cohort-month-based)
window_month = Window.partitionBy('PATIENT_ID').orderBy('COHORT_MONTH').rowsBetween(-sys.maxsize, 0)

# Get LAST_TEST_DATE per patient
df_lab_with_last_date = df_lab.withColumn(
    "LAST_TEST_DATE", 
    F.lead("SVC_DT", 1).over(Window.partitionBy("PATIENT_ID").orderBy("SVC_DT"))
).withColumn("LAST_TEST_DATE", F.coalesce(col("LAST_TEST_DATE"), F.to_date(F.lit(end_date_limit))))

# Generate cohort-month sequence from SVC_DT to LAST_TEST_DATE
# Uses assign_cohort_month() from config to shift by offset
df_lab_with_month_seq = df_lab_with_last_date.withColumn(
    "MONTH_SEQ",
    F.expr(f"""
        sequence(
            {_cohort_month_sql('SVC_DT')},
            {_cohort_month_sql('LAST_TEST_DATE')},
            interval 1 month
        )
    """)
)

# Explode the month sequence to get each cohort-month in the range
df_lab_expanded = df_lab_with_month_seq.withColumn(
    "COHORT_MONTH",
    F.explode("MONTH_SEQ")
).drop("MONTH_SEQ")

# Forward-fill test stages across cohort-months
filled_gfr = F.last(col("GFR_STAGE"), ignorenulls=True).over(window_month)
filled_uacr = F.last(col("UACR_STAGE"), ignorenulls=True).over(window_month)

df_lab_filled = df_lab_expanded.withColumn("GFR_STAGE", filled_gfr).withColumn("UACR_STAGE", filled_uacr)

# Aggregate to remove duplicates per patient-cohort-month and take the max stages
df_lab_clean = df_lab_filled.groupBy("PATIENT_ID", "COHORT_MONTH").agg(
    F.max("GFR_STAGE").alias("GFR_STAGE"),
    F.max("UACR_STAGE").alias("UACR_STAGE")
).orderBy("PATIENT_ID", "COHORT_MONTH")

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
df_lab_clean = df_lab_clean.join(
    df_cohort_ref.select("cohort_ref_month", "cohort_label"),
    df_lab_clean.COHORT_MONTH == df_cohort_ref.cohort_ref_month,
    "left"
).drop("cohort_ref_month")

df_lab_clean.display()

In [0]:
df_lab_clean = df_lab_clean.withColumnRenamed("COHORT_MONTH", "DATE")

In [0]:
df_lab_clean.count(), len(df_lab_clean.columns)

In [0]:
# ============================================================
# COMPARISON: offset=0 validation vs production (same patients)
# ============================================================
base = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output"
import os as _os
_prod_path = f"{base}/kerendia_nba_3_0_lab_journey_output"

def _compare_with_prod():
  prod_lab = spark.read.format("delta").load(_prod_path)
  val_patients = [r.PATIENT_ID for r in df_lab_clean.select("PATIENT_ID").distinct().collect()]
  prod_lab_f = prod_lab.filter(col("PATIENT_ID").isin(val_patients))
  _run_comparison(prod_lab_f)

def _run_comparison(prod_lab_f):
  exec(_COMPARISON_BODY, globals(), {"prod_lab_f": prod_lab_f})

_COMPARISON_BODY = f'''
val_count = df_lab_clean.count()
prod_count = prod_lab_f.count()
print(f"Validation (offset=0): {val_count:,}")
print(f"Production (same pts): {prod_count:,}")
print(f"Difference:            {val_count - prod_count:+,}")

# Key comparisons
val_keys = df_lab_clean.select("PATIENT_ID", "DATE")
prod_keys = prod_lab_f.select("PATIENT_ID", "DATE")

only_in_val = val_keys.subtract(prod_keys)
only_in_prod = prod_keys.subtract(val_keys)
print(f"\nRows only in validation: {only_in_val.count()}")
print(f"Rows only in production: {only_in_prod.count()}")

# Attribute: how many prod-only rows are before DATA_START_DATE?
prod_only_before = only_in_prod.filter(col("DATE") < DATA_START_DATE).count()
prod_only_after = only_in_prod.filter(col("DATE") >= DATA_START_DATE).count()
print(f"  Prod-only before {DATA_START_DATE}: {prod_only_before} (DATA_START_DATE filter)")
print(f"  Prod-only on/after {DATA_START_DATE}: {prod_only_after}")

# Stage differences on overlapping rows
common = df_lab_clean.alias("v").join(
    prod_lab_f.alias("p"),
    (col("v.PATIENT_ID") == col("p.PATIENT_ID")) & (col("v.DATE") == col("p.DATE")),
    "inner"
)
stage_diffs = common.filter(
    (col("v.GFR_STAGE") != col("p.GFR_STAGE")) |
    (col("v.UACR_STAGE") != col("p.UACR_STAGE")) |
    (col("v.GFR_STAGE").isNull() != col("p.GFR_STAGE").isNull()) |
    (col("v.UACR_STAGE").isNull() != col("p.UACR_STAGE").isNull())
)
print(f"\nOverlapping rows: {common.count():,}")
print(f"Stage value differences: {stage_diffs.count()}")
if stage_diffs.count() > 0:
    print("\nSample diffs:")
    stage_diffs.select(
        col("v.PATIENT_ID"), col("v.DATE"),
        col("v.GFR_STAGE").alias("val_GFR"), col("p.GFR_STAGE").alias("prod_GFR"),
        col("v.UACR_STAGE").alias("val_UACR"), col("p.UACR_STAGE").alias("prod_UACR")
    ).orderBy("PATIENT_ID", "DATE").show(20, truncate=False)
'''

# Optional check: the 3.0 production lab journey table may not exist after a data refresh
if _os.path.exists(_prod_path):
  _compare_with_prod()
else:
  print(f"SKIP comparison: production table not found at {_prod_path}")

In [0]:
df_lab.count(), len(df_lab.columns)

In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_lab_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_lab_journey_validation")